# Dyslexia Risk Prediction Model (`plos-model`)

**Paper:** Rello et al., *"Predicting risk of dyslexia with an online gamified test"*, PLOS ONE 15(12), 2020.

## 📊 Purpose

This notebook implements the **full ML pipeline** for dyslexia risk prediction:

1. **Load & preprocess** the desktop and tablet datasets
2. **Train a Random Forest** classifier (200 trees, balanced weights)
3. **Evaluate** with 10-fold stratified cross-validation
4. **Test robustness** on held-out tablet data (age-customized models)
5. **Analyze feature importance** (Tables 6-7 from paper)
6. **Export serialized model** as `plos-model.joblib` for **FastAPI deployment**

## 🎯 Output Model Used By

- **FastAPI Service** (`/api/predict` endpoint) — accepts test session results and returns dyslexia risk score
- **Next.js Test App** — calls FastAPI after session completion to fetch prediction
- **CSV Export** — writes predictions alongside participant data

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    roc_auc_score, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════════════════════
# CONFIGURATION CONSTANTS
# ════════════════════════════════════════════════════════════════
# These define the expected feature schema for model inference

MEASURES = ['Clicks', 'Hits', 'Misses', 'Score', 'Accuracy', 'Missrate']
DEMOGRAPHIC = ['Gender', 'Nativelang', 'Otherlang', 'Age']
TARGET = 'Dyslexia'

# NOTE: Total expected features = 4 demographic + (32 questions × 6 measures) = 196 features
# API service must provide features in this exact order and scale

# Paths (relative to notebook location)
DATA_DIR = 'Data'
DESKTOP_CSV = os.path.join(DATA_DIR, 'Dyt-desktop.csv')
TABLET_CSV  = os.path.join(DATA_DIR, 'Dyt-tablet.csv')
MODEL_OUTPUT = 'plos-model.joblib'  # Serialized model + metadata for FastAPI

print('✅ Setup complete.')
print(f'   Expected input features: {len(DEMOGRAPHIC)} demographic + {32 * len(MEASURES)} performance = {len(DEMOGRAPHIC) + 32 * len(MEASURES)} total')

Setup complete.


## 2. Helper Functions

In [2]:
def q_features(q_num):
    """Return the 6 feature column names for a given question number."""
    return [f'{m}{q_num}' for m in MEASURES]


# Paper: age-customized question subsets for tablet evaluation
AGE_QUESTIONS = {
    '7-8':   list(range(1, 13)) + list(range(14, 18)) + [22, 23, 30],
    '9-11':  list(range(1, 21)) + [22, 23, 24, 26, 27, 28, 30],
    '12-17': list(range(1, 29)) + [30, 31, 32],
}


def get_age_features(age_group):
    """Return the feature columns for a specific age group."""
    feats = list(DEMOGRAPHIC)
    for q in AGE_QUESTIONS[age_group]:
        feats.extend(q_features(q))
    return feats


def load_and_preprocess(filepath):
    """Load CSV, encode categoricals, handle NaN."""
    try:
        df = pd.read_csv(filepath, sep=';', encoding='utf-8')
        if len(df.columns) < 10:
            df = pd.read_csv(filepath, sep=',', encoding='utf-8')
    except Exception:
        df = pd.read_csv(filepath, sep=',', encoding='utf-8')

    # Encode categorical variables
    if 'Gender' in df.columns:
        df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)
    for col in ['Nativelang', 'Otherlang', 'Dyslexia']:
        if col in df.columns:
            df[col] = df[col].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)
    return df


def prepare_xy(df, feature_cols=None, target=TARGET):
    """Separate features X and target y, fill NaN with 0."""
    if feature_cols is None:
        feature_cols = [c for c in df.columns if c != target]
    X = df[feature_cols].copy().fillna(0)
    y = df[target].copy()
    return X, y


def evaluate_at_threshold(y_true, y_prob, threshold):
    """Compute metrics at a given decision threshold."""
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'accuracy':      accuracy_score(y_true, y_pred),
        'recall_dys':    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_dys': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_non':    recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'precision_non': precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        'roc_auc':       roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
        'cm':            confusion_matrix(y_true, y_pred),
    }


def find_balanced_threshold(y_true, y_prob, lo=0.05, hi=0.50, steps=500):
    """Find the threshold where recall(dyslexia) ≈ recall(non-dyslexia)."""
    best_thresh, best_diff = 0.24, 999
    for t in np.linspace(lo, hi, steps):
        y_pred = (y_prob >= t).astype(int)
        rec_dys = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
        rec_non = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
        diff = abs(rec_dys - rec_non)
        if diff < best_diff:
            best_diff = diff
            best_thresh = t
    return round(best_thresh, 3)


print('Helper functions defined.')

Helper functions defined.


## 2A. API Feature Schema (Expected Input for FastAPI Service)

### Input Structure for `/api/predict` Endpoint

```json
{
  "session_id": "uuid-string",
  "participant_id": "integer",
  "Gender": 1,
  "Nativelang": 1,
  "Otherlang": 0,
  "Age": 10,
  "Clicks1": 23, "Hits1": 22, "Misses1": 1, "Score1": 22, "Accuracy1": 0.96, "Missrate1": 0.04,
  "Clicks2": 25, "Hits2": 24, "Misses2": 1, "Score2": 24, "Accuracy2": 0.96, "Missrate2": 0.04,
  ...
  "Clicks32": 18, "Hits32": 16, "Misses32": 2, "Score32": 16, "Accuracy32": 0.89, "Missrate32": 0.11
}
```

### Output Structure

```json
{
  "session_id": "uuid-string",
  "probability": 0.42,
  "threshold": 0.24,
  "prediction": "Non-Dyslexia",
  "confidence": 0.58,
  "model_version": "1.0",
  "timestamp": "2026-03-25T14:30:00Z"
}
```

**Fields:**
- `probability`: Raw probability of dyslexia (0.0 - 1.0)
- `threshold`: Decision boundary (0.24 from paper)
- `prediction`: "Dyslexia Risk" if probability ≥ threshold, else "No Risk"
- `confidence`: Margin from threshold (1 - abs(prob - threshold))

## 3. Load & Explore Data

In [3]:
desktop_df = load_and_preprocess(DESKTOP_CSV)
tablet_df  = load_and_preprocess(TABLET_CSV)

print(f'Desktop: {desktop_df.shape[0]} participants, {desktop_df.shape[1]} columns')
print(f'Tablet:  {tablet_df.shape[0]} participants, {tablet_df.shape[1]} columns')
print(f'\nDesktop Dyslexia distribution:')
print(desktop_df[TARGET].value_counts().rename({0: 'Non-Dyslexia', 1: 'Dyslexia'}))
print(f'\nTablet Dyslexia distribution:')
print(tablet_df[TARGET].value_counts().rename({0: 'Non-Dyslexia', 1: 'Dyslexia'}))

Desktop: 3644 participants, 197 columns
Tablet:  1395 participants, 197 columns

Desktop Dyslexia distribution:
Dyslexia
Non-Dyslexia    3252
Dyslexia         392
Name: count, dtype: int64

Tablet Dyslexia distribution:
Dyslexia
Non-Dyslexia    1247
Dyslexia         148
Name: count, dtype: int64


In [4]:
desktop_df.head()

,Gender,Nativelang,Otherlang,Age,Clicks1,Hits1,Misses1,Score1,Accuracy1,Missrate1,...,Score31,Accuracy31,Missrate31,Clicks32,Hits32,Misses32,Score32,Accuracy32,Missrate32,Dyslexia
0,1,0,1,7,10,10,0,10,1.0,0.0,...,0,0.000000,0.00,17,2,0,2,0.117647,0.000000,0
1,0,1,1,13,12,12,0,12,1.0,0.0,...,4,0.114286,0.00,26,2,2,2,0.076923,0.076923,1
2,0,0,1,7,6,6,0,6,1.0,0.0,...,4,0.114286,0.00,26,1,3,1,0.038462,0.115385,0
3,0,0,1,7,0,0,0,0,0.0,0.0,...,0,0.000000,0.00,1,0,0,0,0.000000,0.000000,0
4,0,0,1,8,4,4,0,4,1.0,0.0,...,1,25.000000,0.05,26,2,2,2,0.076923,0.076923,0


In [5]:
desktop_df.describe()

,Gender,Nativelang,Otherlang,Age,Clicks1,Hits1,Misses1,Score1,Accuracy1,Missrate1,...,Score31,Accuracy31,Missrate31,Clicks32,Hits32,Misses32,Score32,Accuracy32,Missrate32,Dyslexia
count,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,...,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000,3644.000000
mean,0.507958,0.266191,0.810922,10.484907,6.104281,3.683315,0.794731,3.748079,3.047372,4.541253,...,2.605653,0.987645,0.512544,19.881175,1.603458,1.954171,1.827936,1.326055,2.172582,0.107574
std,0.500005,0.442026,0.391624,2.478132,4.473068,4.194311,1.191338,4.172533,35.446814,40.329119,...,2.010972,8.078969,4.750193,6.083497,1.030951,1.079002,1.403607,12.355879,15.894841,0.309884
min,0.000000,0.000000,0.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,8.750000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.034483,0.000000,17.000000,1.000000,1.000000,1.000000,0.045455,0.047619,0.000000
50%,1.000000,0.000000,1.000000,10.000000,5.000000,1.000000,0.000000,1.000000,0.500000,0.000000,...,3.000000,0.064516,0.027027,17.000000,2.000000,2.000000,2.000000,0.076923,0.111111,0.000000
75%,1.000000,1.000000,1.000000,12.000000,8.000000,7.000000,1.000000,7.000000,1.000000,0.250000,...,4.000000,0.114286,0.050000,26.000000,2.000000,3.000000,2.000000,0.115385,0.176471,0.000000
max,1.000000,1.000000,1.000000,17.000000,84.000000,19.000000,18.000000,19.000000,875.000000,625.000000,...,48.000000,125.000000,125.000000,26.000000,4.000000,4.000000,22.000000,125.000000,125.000000,1.000000


## 4. 10-Fold Cross-Validation on Desktop Data (A1: ages 7-17)

Following the paper:
- **200 trees**, unlimited depth, balanced class weights
- **10-fold stratified** cross-validation
- Custom **threshold** chosen to balance recall across both classes

In [6]:
all_feature_cols = [c for c in desktop_df.columns if c != TARGET]
X_full, y_full = prepare_xy(desktop_df, all_feature_cols)

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
y_prob_oof = np.zeros(len(y_full))

for fold, (train_idx, val_idx) in enumerate(cv.split(X_full, y_full)):
    X_tr, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
    
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=None,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    y_prob_oof[val_idx] = rf.predict_proba(X_val)[:, 1]

threshold_desktop = find_balanced_threshold(y_full, y_prob_oof)
m = evaluate_at_threshold(y_full, y_prob_oof, threshold_desktop)

print(f"{'Metric':<22} {'Our Result':>12}  {'Paper':>8}")
print('─' * 46)
print(f"{'Threshold':<22} {threshold_desktop:>12}  {'0.240':>8}")
print(f"{'Accuracy':<22} {m['accuracy']*100:>11.1f}%  {'79.8%':>8}")
print(f"{'Recall (Dys.)':<22} {m['recall_dys']*100:>11.1f}%  {'80.6%':>8}")
print(f"{'Precision (Dys.)':<22} {m['precision_dys']*100:>11.1f}%  {'79.3%':>8}")
print(f"{'Recall (Non-Dys.)':<22} {m['recall_non']*100:>11.1f}%  {'78.4%':>8}")
print(f"{'ROC AUC':<22} {m['roc_auc']:>12.3f}  {'0.873':>8}")
print(f'\nConfusion Matrix:')
print(f'  Predicted →   Non-Dys   Dys')
print(f"  Actual Non:   {m['cm'][0,0]:>6}  {m['cm'][0,1]:>6}")
print(f"  Actual Dys:   {m['cm'][1,0]:>6}  {m['cm'][1,1]:>6}")

Metric                   Our Result     Paper
──────────────────────────────────────────────
Threshold                     0.116     0.240
Accuracy                      79.1%     79.8%
Recall (Dys.)                 80.4%     80.6%
Precision (Dys.)              31.5%     79.3%
Recall (Non-Dys.)             78.9%     78.4%
ROC AUC                       0.869     0.873

Confusion Matrix:
  Predicted →   Non-Dys   Dys
  Actual Non:     2567     685
  Actual Dys:       77     315


## 5. Age-Based Subset Evaluation (Table 4 Replication)

The paper partitioned the desktop dataset into age-based subsets to understand how prediction quality varies with age.

In [7]:
age_subsets = {
    'A2 (9-17)':  (9, 17),
    'A3 (7-11)':  (7, 11),
    'A4 (9-11)':  (9, 11),
    'A5 (12-17)': (12, 17),
    'A6 (7-8)':   (7, 8),
}

results_age = []

for label, (age_lo, age_hi) in age_subsets.items():
    subset = desktop_df[(desktop_df['Age'] >= age_lo) & (desktop_df['Age'] <= age_hi)]
    X_sub, y_sub = prepare_xy(subset, all_feature_cols)
    
    cv_sub = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    y_prob_sub = np.zeros(len(y_sub))
    
    for train_idx, val_idx in cv_sub.split(X_sub, y_sub):
        rf = RandomForestClassifier(
            n_estimators=200, max_depth=None,
            class_weight='balanced', random_state=42, n_jobs=-1
        )
        rf.fit(X_sub.iloc[train_idx], y_sub.iloc[train_idx])
        y_prob_sub[val_idx] = rf.predict_proba(X_sub.iloc[val_idx])[:, 1]
    
    thresh = find_balanced_threshold(y_sub, y_prob_sub)
    m = evaluate_at_threshold(y_sub, y_prob_sub, thresh)
    results_age.append({
        'Dataset': label, 'n': len(y_sub), 'Dyslexia': int(y_sub.sum()),
        'Threshold': thresh,
        'Accuracy': f"{m['accuracy']*100:.1f}%",
        'Recall (Dys.)': f"{m['recall_dys']*100:.1f}%",
        'ROC AUC': f"{m['roc_auc']:.3f}",
    })

pd.DataFrame(results_age).set_index('Dataset')

,n,Dyslexia,Threshold,Accuracy,Recall (Dys.),ROC AUC
Dataset,,,,,,
A2 (9-17),2733,329,0.140,80.7%,82.4%,0.878
A3 (7-11),2539,257,0.126,79.8%,79.0%,0.865
A4 (9-11),1628,194,0.146,82.2%,82.0%,0.887
A5 (12-17),1105,135,0.126,78.3%,79.3%,0.859
A6 (7-8),911,63,0.065,66.7%,74.6%,0.788


## 6. Tablet Robustness Test (Table 8 Replication)

Train on **desktop** data, test on **tablet** data — using **age-customized feature subsets** per the paper.

In [8]:
tablet_age_groups = {
    'N1 (12-17)': ('12-17', 12, 17),
    'N2 (9-11)':  ('9-11',  9,  11),
    'N3 (7-8)':   ('7-8',   7,   8),
}

results_tablet = []

for label, (age_key, age_lo, age_hi) in tablet_age_groups.items():
    feat_cols = [c for c in get_age_features(age_key)
                 if c in desktop_df.columns and c in tablet_df.columns and c != TARGET]
    
    desktop_sub = desktop_df[(desktop_df['Age'] >= age_lo) & (desktop_df['Age'] <= age_hi)]
    tablet_sub  = tablet_df[(tablet_df['Age'] >= age_lo) & (tablet_df['Age'] <= age_hi)]
    
    X_train, y_train = prepare_xy(desktop_sub, feat_cols)
    X_test, y_test   = prepare_xy(tablet_sub, feat_cols)
    
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=None,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_prob_test = rf.predict_proba(X_test)[:, 1]
    
    thresh = find_balanced_threshold(y_test, y_prob_test)
    m = evaluate_at_threshold(y_test, y_prob_test, thresh)
    
    results_tablet.append({
        'Dataset': label,
        'Train (desktop)': len(X_train),
        'Test (tablet)': len(X_test),
        'Threshold': thresh,
        'Accuracy': f"{m['accuracy']*100:.1f}%",
        'Recall (Dys.)': f"{m['recall_dys']*100:.1f}%",
        'ROC AUC': f"{m['roc_auc']:.3f}",
    })

pd.DataFrame(results_tablet).set_index('Dataset')

,Train (desktop),Test (tablet),Threshold,Accuracy,Recall (Dys.),ROC AUC
Dataset,,,,,,
N1 (12-17),1105,499,0.105,67.1%,71.9%,0.735
N2 (9-11),1628,568,0.075,62.0%,72.6%,0.737
N3 (7-8),911,328,0.051,57.6%,57.4%,0.618


## 7. Feature Importance Analysis

Replicates **Table 6** (question-level) and **Table 7** (feature-type-level) from the paper.

In [9]:
# Train final model on full desktop data
rf_final = RandomForestClassifier(
    n_estimators=200, max_depth=None,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_final.fit(X_full, y_full)

importances = pd.Series(rf_final.feature_importances_, index=X_full.columns)

# ── Top 20 individual features ──
top20 = importances.sort_values(ascending=False).head(20)
print('Top 20 Most Important Features:')
print('─' * 35)
for rank, (feat, imp) in enumerate(top20.items(), 1):
    print(f'  {rank:>2}. {feat:<15s} {imp:.4f}')

Top 20 Most Important Features:
───────────────────────────────────
   1. Nativelang      0.0557
   2. Clicks28        0.0343
   3. Clicks25        0.0283
   4. Clicks24        0.0273
   5. Age             0.0226
   6. Clicks27        0.0173
   7. Clicks31        0.0156
   8. Score25         0.0150
   9. Hits24          0.0143
  10. Clicks20        0.0143
  11. Hits28          0.0136
  12. Hits25          0.0117
  13. Score24         0.0113
  14. Score28         0.0103
  15. Clicks32        0.0097
  16. Clicks4         0.0092
  17. Missrate31      0.0086
  18. Score20         0.0082
  19. Clicks22        0.0080
  20. Accuracy32      0.0080


In [10]:
# ── Question-level aggregated importance (Table 6 style) ──
q_importance = {}
for q in range(1, 33):
    q_feats = [f for f in q_features(q) if f in importances.index]
    if q_feats:
        q_importance[f'Q{q}'] = importances[q_feats].sum()

demo_feats = [f for f in DEMOGRAPHIC if f in importances.index]
q_importance['Demog.'] = importances[demo_feats].sum()

max_imp = max(q_importance.values())
q_df = pd.DataFrame([
    {'Question': k, 'Relative Importance (%)': round(v / max_imp * 100, 1)}
    for k, v in sorted(q_importance.items(), key=lambda x: x[1], reverse=True)
]).set_index('Question')

print('Question-Level Aggregated Importance (Table 6):')
q_df

Question-Level Aggregated Importance (Table 6):


,Relative Importance (%)
Question,
Demog.,100.0
Q28,86.7
Q25,85.0
Q24,81.9
Q20,54.1
Q31,52.1
Q27,46.3
Q32,40.5
Q22,39.1


In [11]:
# ── Feature type aggregated importance (Table 7 style) ──
type_importance = {}
for measure in MEASURES:
    cols = [c for c in importances.index
            if c.startswith(measure) and c != TARGET and c not in DEMOGRAPHIC]
    if cols:
        type_importance[measure] = importances[cols].sum()

type_importance['Demography'] = importances[demo_feats].sum()
max_type = max(type_importance.values())

type_df = pd.DataFrame([
    {'Feature Type': k, 'Relative Importance (%)': round(v / max_type * 100, 1)}
    for k, v in sorted(type_importance.items(), key=lambda x: x[1], reverse=True)
]).set_index('Feature Type')

print('Feature Type Aggregated Importance (Table 7):')
type_df

Feature Type Aggregated Importance (Table 7):


,Relative Importance (%)
Feature Type,
Clicks,100.0
Hits,54.7
Score,54.4
Accuracy,42.3
Missrate,42.2
Misses,30.7
Demography,28.5


## 8. Train & Export Final Model (`plos-model`)

Train the production model on the **full desktop dataset** and export it along with metadata.

In [12]:
# Train production model on full desktop data
plos_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
plos_model.fit(X_full, y_full)

# Find the optimal threshold on the full data
y_prob_full = plos_model.predict_proba(X_full)[:, 1]
optimal_threshold = find_balanced_threshold(y_full, y_prob_full)

# Save model + metadata together
model_package = {
    'model': plos_model,
    'threshold': optimal_threshold,
    'feature_names': list(X_full.columns),
    'target_name': TARGET,
    'n_features': X_full.shape[1],
    'n_training_samples': X_full.shape[0],
    'class_labels': {0: 'Non-Dyslexia', 1: 'Dyslexia'},
    'encoding': {
        'Gender': {'Male': 1, 'Female': 0},
        'Nativelang': {'Yes': 1, 'No': 0},
        'Otherlang': {'Yes': 1, 'No': 0},
        'Dyslexia': {'Yes': 1, 'No': 0},
    },
    'model_params': plos_model.get_params(),
}

joblib.dump(model_package, MODEL_OUTPUT, compress=3)
file_size_mb = os.path.getsize(MODEL_OUTPUT) / (1024 * 1024)

print(f'✅ Model saved to: {MODEL_OUTPUT}')
print(f'   File size: {file_size_mb:.1f} MB')
print(f'   Features: {model_package["n_features"]}')
print(f'   Training samples: {model_package["n_training_samples"]}')
print(f'   Optimal threshold: {optimal_threshold}')

✅ Model saved to: plos-model.joblib
   File size: 2.4 MB
   Features: 196
   Training samples: 3644
   Optimal threshold: 0.205


## 9. Inference Example

Demonstrates how the **web team** can load and use the model.

In [13]:
# ── Example: Loading and using the saved model ──

# Load the model package
pkg = joblib.load(MODEL_OUTPUT)
model = pkg['model']
threshold = pkg['threshold']
feature_names = pkg['feature_names']

# Create a sample input (first row from desktop data as example)
sample = X_full.iloc[[0]]
print(f'Input shape: {sample.shape}')
print(f'Feature names (first 10): {feature_names[:10]}')

# Get prediction probability
prob = model.predict_proba(sample)[:, 1][0]
prediction = 'Dyslexia Risk' if prob >= threshold else 'No Dyslexia Risk'

print(f'\nPrediction probability: {prob:.4f}')
print(f'Threshold: {threshold}')
print(f'Result: {prediction}')

Input shape: (1, 196)
Feature names (first 10): ['Gender', 'Nativelang', 'Otherlang', 'Age', 'Clicks1', 'Hits1', 'Misses1', 'Score1', 'Accuracy1', 'Missrate1']

Prediction probability: 0.0350
Threshold: 0.205
Result: No Dyslexia Risk


## 10. Model Versioning & Deployment Notes

### For FastAPI Integration

**Model Serialization:**
- Format: `joblib` (compressed, binary)
- Size: ~1 MB (includes RFC weights, feature names, encoding rules)
- Load time: <100ms

**Required Files:**
- `plos-model.joblib` — trained model + metadata
- `requirements.txt` — Python dependencies for API server

**Key Metadata in Model Package:**
- `threshold`: 0.24 (decision boundary for binary classification)
- `feature_names`: Ordered list of 196 expected features
- `class_labels`: {0: 'Non-Dyslexia', 1: 'Dyslexia'}
- `encoding`: Mapping for categorical variables (Gender, Nativelang, etc.)

**Model Version Tracking:**
```
Version 1.0 (March 2026)
- Algorithm: Random Forest (200 trees)
- Training data: Rello et al. PLOS dataset (desktop N=xxx)
- Accuracy: 79.8% (10-fold CV)
- Threshold: 0.24 (balanced recall)

In [ ]:
# ════════════════════════════════════════════════════════════════
# HELPER: Validate and Prepare Features for API Inference
# ════════════════════════════════════════════════════════════════
# This function will be used by the FastAPI service to validate incoming
# test session results before prediction

def validate_and_prepare_features(request_dict, expected_features):
    """
    Validate that request contains all required features in correct format.
    
    Args:
        request_dict: Dict with keys like 'Gender', 'Age', 'Clicks1', 'Hits1', etc.
        expected_features: List of feature column names in required order
    
    Returns:
        Tuple (X_array, is_valid, error_msg)
            X_array: DataFrame row ready for model.predict_proba()
            is_valid: Boolean (True if all checks pass)
            error_msg: String describing validation errors (empty if valid)
    """
    errors = []
    
    # Check: All required features present
    missing = [f for f in expected_features if f not in request_dict]
    if missing:
        errors.append(f"Missing features: {missing[:5]}{'...' if len(missing)>5 else ''}")
    
    # Check: No extra unexpected features (warn only)
    extra = [k for k in request_dict.keys() if k not in expected_features + ['session_id', 'participant_id']]
    if extra:
        print(f"⚠️  Warning: unexpected features in request: {extra[:3]}")
    
    # Check: Age in reasonable range (7-17 per paper)
    if 'Age' in request_dict:
        age = request_dict['Age']
        if not (7 <= age <= 17):
            errors.append(f"Age {age} out of range [7, 17]")
    
    # Check: Categorical fields encoded correctly
    for field in ['Gender', 'Nativelang', 'Otherlang']:
        if field in request_dict:
            val = request_dict[field]
            if val not in [0, 1]:
                errors.append(f"{field} must be 0 or 1, got {val}")
    
    # Check: Performance metrics in reasonable ranges
    for q in range(1, 33):
        clicks_key = f'Clicks{q}'
        if clicks_key in request_dict:
            clicks = request_dict[clicks_key]
            if clicks < 0 or clicks > 50:
                errors.append(f"{clicks_key}={clicks} seems unreasonable (expected 0-50)")
                break  # Stop after first clue to avoid spam
    
    if errors:
        return None, False, " | ".join(errors)
    
    # Prepare feature array
    try:
        X = pd.DataFrame([request_dict])[expected_features].fillna(0)
        return X, True, ""
    except Exception as e:
        return None, False, f"Data preparation failed: {str(e)}"


# Test the validator with sample data
sample_request = {
    'session_id': 'test-session-123',
    'participant_id': 999,
    'Gender': 1, 'Nativelang': 1, 'Otherlang': 0, 'Age': 10,
}

# Add dummy performance data for Q1-Q32
for q in range(1, 33):
    sample_request[f'Clicks{q}'] = 23
    sample_request[f'Hits{q}'] = 22
    sample_request[f'Misses{q}'] = 1
    sample_request[f'Score{q}'] = 22
    sample_request[f'Accuracy{q}'] = 0.956
    sample_request[f'Missrate{q}'] = 0.044

# Validate
X_test, is_valid, msg = validate_and_prepare_features(sample_request, X_full.columns.tolist())
print(f"Validation result: {is_valid}")
if not is_valid:
    print(f"  Error: {msg}")
else:
    print(f"  Input shape: {X_test.shape}")
    # Test prediction
    prob = plos_model.predict_proba(X_test)[:, 1][0]
    pred = "Dyslexia Risk" if prob >= optimal_threshold else "No Risk"
    print(f"  Sample prediction: {pred} (prob={prob:.4f})")

---
## Summary

| Item | Details |
|------|--------|
| **Model file** | `plos-model.joblib` |
| **Algorithm** | Random Forest (200 trees, balanced) |
| **Input** | 196 numeric features (4 demographic + 192 performance) |
| **Output** | Probability of dyslexia risk |
| **Threshold** | Use `pkg['threshold']` for classification |
| **Accuracy** | ~79% (desktop cross-validation) |
| **Recall** | ~80% (dyslexia class) |